# BAREC Model Evaluation Comparison

This notebook evaluates multiple PIXEL models with different Arabic processing configurations on the BAREC sentence-level readability dataset. It compares performance across validation and test splits and presents results in comprehensive tables.

## Setup and Configuration

In [1]:
import sys
import os
import logging
import pandas as pd
import numpy as np
import torch
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add project path
sys.path.append('/home/bens/pixel')

from src.pixel import (
    PIXELForSequenceClassification,
    PangoCairoTextRenderer,
    BARECDataset,
    Modality,
    get_transforms
)
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, cohen_kappa_score, mean_absolute_error

# Set up logging
logging.basicConfig(level=logging.WARNING)  # Reduce noise
logger = logging.getLogger(__name__)

print("✅ All imports successful!")
print(f"🔧 Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

/opt/anaconda3/envs/pixel-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!
🔧 Device: CUDA


## Model Configuration Setup

Define the models and their corresponding processing configurations to evaluate:

In [2]:
@dataclass
class ModelConfig:
    """Configuration for a model to evaluate"""
    name: str
    model_path: str
    processing_config: Optional[str]
    description: str
    renderer_path: str = "Team-PIXEL/pixel-base"


"""
no-unicode-normalize
arabic-default
arabic-norm-dediac
arabic-nonorm-diac
buckwalter-default
buckwalter-norm-dediac
buckwalter-nonorm-diac
hsb-default
hsb-norm-dediac
hsb-nonorm-diac
"""

# Define models to evaluate
MODEL_CONFIGS = [
    ModelConfig(
        name="Arabic-No-Unicode-Normalize",
        model_path="../runs/pixel-base-no-unicode-normalize-256-64-1-5e-05-25000-42",
        processing_config="no-unicode-normalize",
        description="Arabic text without Unicode normalization"
    ),
    ModelConfig(
        name="Arabic-Default",
        model_path="../runs/pixel-base-arabic-default-256-64-1-5e-05-25000-42",
        processing_config="arabic-default",
        description="PIXEL with Arabic default processing"
    ),
    ModelConfig(
        name="Arabic-Norm-Dediac",
        model_path="../runs/pixel-base-arabic-norm-dediac-256-64-1-5e-05-25000-42",
        processing_config="arabic-norm-dediac",
        description="PIXEL with Arabic normalization and diacritics removal"
    ),
    ModelConfig(
        name="Arabic-Nonorm-Diac",
        model_path="../runs/pixel-base-arabic-nonorm-diac-256-64-1-5e-05-25000-42",
        processing_config="arabic-nonorm-diac",
        description="PIXEL with Arabic non-normalized diacritics"
    ),
    ModelConfig(
        name="Buckwalter-Default",
        model_path="../runs/pixel-base-buckwalter-default-256-64-1-5e-05-25000-42",
        processing_config="buckwalter-default",
        description="PIXEL with Buckwalter default processing"
    ),
    ModelConfig(
        name="Buckwalter-Norm-Dediac",
        model_path="../runs/pixel-base-buckwalter-norm-dediac-256-64-1-5e-05-25000-42",
        processing_config="buckwalter-norm-dediac",
        description="PIXEL with Buckwalter normalization and diacritics removal"
    ),
    ModelConfig(
        name="Buckwalter-Nonorm-Diac",
        model_path="../runs/pixel-base-buckwalter-nonorm-diac-256-64-1-5e-05-25000-42",
        processing_config="buckwalter-nonorm-diac",
        description="PIXEL with Buckwalter non-normalized diacritics"
    ),
    ModelConfig(
        name="HSB-Default",
        model_path="../runs/pixel-base-hsb-default-256-64-1-5e-05-25000-42",
        processing_config="hsb-default",
        description="PIXEL with HSB default processing"
    ),
    ModelConfig(
        name="HSB-Norm-Dediac",
        model_path="../runs/pixel-base-hsb-norm-dediac-256-64-1-5e-05-25000-42",
        processing_config="hsb-norm-dediac",
        description="PIXEL with HSB normalization and diacritics removal"
    ),
    ModelConfig(
        name="HSB-Nonorm-Diac",
        model_path="../runs/pixel-base-hsb-nonorm-diac-256-64-1-5e-05-25000-42",
        processing_config="hsb-nonorm-diac",
        description="PIXEL with HSB non-normalized diacritics"
    ),
]

# Dataset configuration
DATASET_CONFIG = {
    "dataset_name": "CAMeL-Lab/BAREC-Shared-Task-2025-sent",
    "max_seq_length": 256,
    "num_labels": 19,
    "batch_size": 16,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

print(f"📊 Configured {len(MODEL_CONFIGS)} models for evaluation:")
for config in MODEL_CONFIGS:
    print(f"  • {config.name}: {config.description}")
    
print(f"\n🎯 Dataset: {DATASET_CONFIG['dataset_name']}")
print(f"📏 Max sequence length: {DATASET_CONFIG['max_seq_length']}")
print(f"🔢 Batch size: {DATASET_CONFIG['batch_size']}")

📊 Configured 10 models for evaluation:
  • Arabic-No-Unicode-Normalize: Arabic text without Unicode normalization
  • Arabic-Default: PIXEL with Arabic default processing
  • Arabic-Norm-Dediac: PIXEL with Arabic normalization and diacritics removal
  • Arabic-Nonorm-Diac: PIXEL with Arabic non-normalized diacritics
  • Buckwalter-Default: PIXEL with Buckwalter default processing
  • Buckwalter-Norm-Dediac: PIXEL with Buckwalter normalization and diacritics removal
  • Buckwalter-Nonorm-Diac: PIXEL with Buckwalter non-normalized diacritics
  • HSB-Default: PIXEL with HSB default processing
  • HSB-Norm-Dediac: PIXEL with HSB normalization and diacritics removal
  • HSB-Nonorm-Diac: PIXEL with HSB non-normalized diacritics

🎯 Dataset: CAMeL-Lab/BAREC-Shared-Task-2025-sent
📏 Max sequence length: 256
🔢 Batch size: 16


## Utility Functions

Define helper functions for model loading, prediction, and evaluation:

In [3]:
def load_model_and_renderer(model_config: ModelConfig) -> Tuple[PIXELForSequenceClassification, PangoCairoTextRenderer]:
    """
    Load a PIXEL model and its corresponding renderer.
    """
    print(f"🔧 Loading {model_config.name}...")
    
    # Load renderer
    renderer = PangoCairoTextRenderer.from_pretrained(
        model_config.renderer_path,
        rgb=False
    )
    renderer.max_seq_length = DATASET_CONFIG["max_seq_length"]
    
    # Load model
    try:
        model = PIXELForSequenceClassification.from_pretrained(
            model_config.model_path,
            num_labels=DATASET_CONFIG["num_labels"]
        )
    except Exception as e:
        print(f"   ⚠️ Failed to load from {model_config.model_path}, using base model")
        model = PIXELForSequenceClassification.from_pretrained(
            "Team-PIXEL/pixel-base",
            num_labels=DATASET_CONFIG["num_labels"]
        )
    
    model.to(DATASET_CONFIG["device"])
    model.eval()
    
    print(f"   ✅ Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
    return model, renderer

def create_dataset(split: str, renderer: PangoCairoTextRenderer, processing_config: Optional[str]) -> BARECDataset:
    """
    Create a BAREC dataset for the given split and processing configuration.
    """
    transforms = get_transforms(
        do_resize=True,
        size=(renderer.pixels_per_patch, renderer.pixels_per_patch * renderer.max_seq_length),
    )
    
    dataset = BARECDataset(
        dataset_name=DATASET_CONFIG["dataset_name"],
        processor=renderer,
        modality=Modality.IMAGE,
        max_seq_length=DATASET_CONFIG["max_seq_length"],
        split=split,
        transforms=transforms,
        processing_config_name=processing_config,
        # inference=True  # Include IDs for tracking
    )
    
    return dataset

def create_dataloader(dataset: BARECDataset) -> DataLoader:
    """
    Create a DataLoader for the dataset.
    """
    def collate_fn(batch):
        pixel_values = torch.stack([item['pixel_values'] for item in batch])
        attention_mask = torch.stack([item['attention_mask'] for item in batch])
        
        batch_dict = {
            'pixel_values': pixel_values,
            'attention_mask': attention_mask
        }
        
        # Include labels and IDs if available
        if 'label' in batch[0]:
            batch_dict['labels'] = torch.tensor([item['label'] for item in batch], dtype=torch.long)
        if 'id' in batch[0]:
            batch_dict['ids'] = [item['id'] for item in batch]
        
        return batch_dict
    
    return DataLoader(
        dataset,
        batch_size=DATASET_CONFIG["batch_size"],
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True if DATASET_CONFIG["device"] == 'cuda' else False
    )

def make_predictions(model: PIXELForSequenceClassification, dataloader: DataLoader) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """
    Make predictions using the model on the given dataloader.
    """
    model.eval()
    all_predictions = []
    all_probabilities = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            # Move to device
            pixel_values = batch['pixel_values'].to(DATASET_CONFIG["device"])
            attention_mask = batch['attention_mask'].to(DATASET_CONFIG["device"])
            
            # Forward pass
            outputs = model(pixel_values=pixel_values, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Get predictions and probabilities
            probabilities = torch.softmax(logits, dim=-1)
            predictions = torch.argmax(logits, dim=-1)
            
            # Store results
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            
            # Store labels if available
            if 'labels' in batch:
                all_labels.extend(batch['labels'].numpy())
    
    return (
        np.array(all_predictions),
        np.array(all_probabilities),
        np.array(all_labels) if all_labels else None
    )

def calculate_metrics(predictions: np.ndarray, true_labels: np.ndarray) -> Dict[str, float]:
    """
    Calculate evaluation metrics.
    """
    accuracy = accuracy_score(true_labels, predictions)
    accuracy_margin_1 = np.mean(np.abs(predictions - true_labels) <= 1)
    qwk = cohen_kappa_score(true_labels, predictions, weights='quadratic')
    mae = mean_absolute_error(true_labels, predictions)
    
    return {
        'accuracy': accuracy,
        'accuracy_margin_1': accuracy_margin_1,
        'qwk': qwk,
        'mae': mae
    }

print("🔧 Utility functions defined successfully!")

🔧 Utility functions defined successfully!


## Model Evaluation Pipeline

Run evaluation for all models and configurations:

In [4]:
def evaluate_model(model_config: ModelConfig) -> Dict[str, Dict[str, float]]:
    """
    Evaluate a single model configuration on both validation and test splits.
    """
    results = {}
    
    try:
        # Load model and renderer
        model, renderer = load_model_and_renderer(model_config)
        
        # Evaluate on both splits
        for split in ['validation', 'test']:
            print(f"   📊 Evaluating on {split} split...")
            
            # Create dataset and dataloader
            dataset = create_dataset(split, renderer, model_config.processing_config)
            dataloader = create_dataloader(dataset)
            
            print(f"      📦 Loaded {len(dataset)} examples")
            
            # Make predictions
            predictions, probabilities, true_labels = make_predictions(model, dataloader)
            
            # Calculate metrics if labels are available (validation split)
            if true_labels is not None and len(true_labels) > 0:
                metrics = calculate_metrics(predictions, true_labels)
                results[split] = metrics
                
                print(f"      ✅ Accuracy: {metrics['accuracy']:.4f}")
                print(f"      ✅ QWK: {metrics['qwk']:.4f}")
                print(f"      ✅ MAE: {metrics['mae']:.4f}")
            else:
                # For test split without labels, store prediction statistics
                results[split] = {
                    'num_predictions': len(predictions),
                    'mean_prediction': float(np.mean(predictions + 1)),  # Convert to 1-19 scale
                    'std_prediction': float(np.std(predictions + 1)),
                    'mean_confidence': float(np.mean(np.max(probabilities, axis=1)))
                }
                
                print(f"      📊 Generated {len(predictions)} predictions")
                print(f"      📊 Mean prediction: {results[split]['mean_prediction']:.2f}")
                print(f"      📊 Mean confidence: {results[split]['mean_confidence']:.3f}")
        
        # Clean up GPU memory
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"   ❌ Error evaluating {model_config.name}: {str(e)}")
        results = {'validation': {'error': str(e)}, 'test': {'error': str(e)}}
    
    return results

# Run evaluation for all models
print("🚀 Starting model evaluation...")
print("=" * 80)

all_results = {}

for i, model_config in enumerate(MODEL_CONFIGS):
    print(f"\n📋 [{i+1}/{len(MODEL_CONFIGS)}] Evaluating {model_config.name}")
    print(f"   🔗 Model: {model_config.model_path}")
    print(f"   ⚙️ Processing: {model_config.processing_config or 'None (Original)'}")
    
    results = evaluate_model(model_config)
    all_results[model_config.name] = {
        'config': model_config,
        'results': results
    }

print("\n✅ Evaluation completed for all models!")

🚀 Starting model evaluation...

📋 [1/10] Evaluating Arabic-No-Unicode-Normalize
   🔗 Model: ../runs/pixel-base-no-unicode-normalize-256-64-1-5e-05-25000-42
   ⚙️ Processing: no-unicode-normalize
🔧 Loading Arabic-No-Unicode-Normalize...
   ✅ Model loaded: 86,451,475 parameters
   📊 Evaluating on validation split...


100%|██████████| 7310/7310 [00:00<00:00, 9806.25it/s]


      📦 Loaded 7310 examples


Predicting: 100%|██████████| 457/457 [00:37<00:00, 12.03it/s]


      ✅ Accuracy: 0.3874
      ✅ QWK: 0.6507
      ✅ MAE: 1.7811
   📊 Evaluating on test split...


...

      ✅ Accuracy: 0.4063
      ✅ QWK: 0.6704
      ✅ MAE: 1.7053
   📊 Evaluating on test split...


100%|██████████| 7286/7286 [01:39<00:00, 73.08it/s] 


      📦 Loaded 7286 examples


Predicting: 100%|██████████| 456/456 [00:38<00:00, 11.70it/s]


      ✅ Accuracy: 0.4126
      ✅ QWK: 0.6741
      ✅ MAE: 1.6657

✅ Evaluation completed for all models!


## Results Analysis and Visualization

Create comprehensive tables and analysis of the results:

In [17]:
def create_results_tables(all_results: Dict) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Create formatted results tables for validation and test splits.
    """
    
    # Results for both splits (treated equally)
    val_data = []
    test_data = []
    
    for model_name, data in all_results.items():
        config = data['config']
        results = data['results']
        
        # Process both validation and test splits the same way
        for split in ['validation', 'test']:
            if split in results and 'accuracy' in results[split]:
                # Successful evaluation with metrics
                split_metrics = results[split]
                row_data = {
                    'Model': model_name,
                    'Processing Config': config.processing_config or 'Original',
                    'Description': config.description,
                    'Accuracy': f"{split_metrics['accuracy']*100:.1f}",
                    'Accuracy ±1': f"{split_metrics['accuracy_margin_1']*100:.1f}",
                    'MAE': f"{split_metrics['mae']:.2f}",
                    'QWK': f"{split_metrics['qwk']*100:.1f}",
                    'Status': '✅ Success'
                }
            else:
                # Failed evaluation or no metrics
                row_data = {
                    'Model': model_name,
                    'Processing Config': config.processing_config or 'Original',
                    'Description': config.description,
                    'Accuracy': 'N/A',
                    'Accuracy ±1': 'N/A',
                    'QWK': 'N/A',
                    'MAE': 'N/A',
                    'Status': '❌ Error' if 'error' in results.get(split, {}) else '⚠️ No Labels'
                }
            
            # Append to appropriate data array based on split
            if split == 'validation':
                val_data.append(row_data)
            else:  # test
                test_data.append(row_data)
    
    val_df = pd.DataFrame(val_data)
    test_df = pd.DataFrame(test_data)
    
    return val_df, test_df

# Create results tables
validation_df, test_df = create_results_tables(all_results)

print("📊 VALIDATION SPLIT RESULTS")
print("=" * 100)
print(validation_df.to_string(index=False))

print("\n\n📊 TEST SPLIT RESULTS")
print("=" * 100)
print(test_df.to_string(index=False))

📊 VALIDATION SPLIT RESULTS
                      Model      Processing Config                                                Description Accuracy Accuracy ±1  MAE  QWK    Status
Arabic-No-Unicode-Normalize   no-unicode-normalize                  Arabic text without Unicode normalization     38.7        52.5 1.78 65.1 ✅ Success
             Arabic-Default         arabic-default                       PIXEL with Arabic default processing     35.6        49.9 1.84 64.5 ✅ Success
         Arabic-Norm-Dediac     arabic-norm-dediac     PIXEL with Arabic normalization and diacritics removal     38.1        53.7 1.85 63.3 ✅ Success
         Arabic-Nonorm-Diac     arabic-nonorm-diac                PIXEL with Arabic non-normalized diacritics     41.1        54.7 1.78 63.2 ✅ Success
         Buckwalter-Default     buckwalter-default                   PIXEL with Buckwalter default processing     43.8        57.5 1.63 68.0 ✅ Success
     Buckwalter-Norm-Dediac buckwalter-norm-dediac PIXEL with Buckw

## Performance Analysis

Analyze and rank the models by performance:

In [ ]:
def analyze_performance(validation_df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze and rank model performance.
    """
    # Filter successful runs
    successful_df = validation_df[validation_df['Status'] == '✅ Success'].copy()
    
    if len(successful_df) == 0:
        print("⚠️ No successful model evaluations found!")
        return pd.DataFrame()
    
    # Convert metrics to numeric
    for col in ['Accuracy', 'Accuracy ±1', 'QWK', 'MAE']:
        successful_df[col] = pd.to_numeric(successful_df[col], errors='coerce')
    
    # Calculate ranks (lower is better for MAE, higher is better for others)
    successful_df['Accuracy_Rank'] = successful_df['Accuracy'].rank(ascending=False)
    successful_df['QWK_Rank'] = successful_df['QWK'].rank(ascending=False)
    successful_df['MAE_Rank'] = successful_df['MAE'].rank(ascending=True)  # Lower is better
    
    
    # Sort by average rank
    ranked_df = successful_df.sort_values('QWK_Rank')
    
    # Create summary table
    summary_data = []
    for _, row in ranked_df.iterrows():
        summary_data.append({
            'Model': row['Model'],
            'Processing': row['Processing Config'],
            'Accuracy': f"{row['Accuracy']:.4f}",
            'QWK': f"{row['QWK']:.4f}",
            'MAE': f"{row['MAE']:.4f}",
        })
    
    return pd.DataFrame(summary_data)

# Performance analysis
performance_df = analyze_performance(test_df)

if len(performance_df) > 0:
    print("🏆 MODEL PERFORMANCE RANKING")
    print("=" * 80)
    print(performance_df.to_string(index=False))
    
    # Best model analysis
    best_model = performance_df.iloc[0]
    print(f"\n🥇 BEST PERFORMING MODEL")
    print("=" * 40)
    print(f"Model: {best_model['Model']}")
    print(f"Processing: {best_model['Processing']}")
    print(f"Accuracy: {best_model['Accuracy']}")
    print(f"QWK: {best_model['QWK']}")
    print(f"MAE: {best_model['MAE']}")
else:
    print("⚠️ No models to rank - all evaluations failed.")

🏆 MODEL PERFORMANCE RANKING
                      Model             Processing Accuracy     QWK    MAE
     Buckwalter-Nonorm-Diac buckwalter-nonorm-diac  42.8000 68.4000 1.6000
            HSB-Norm-Dediac        hsb-norm-dediac  41.3000 68.1000 1.6800
                HSB-Default            hsb-default  43.4000 67.7000 1.6200
            HSB-Nonorm-Diac        hsb-nonorm-diac  41.3000 67.4000 1.6700
         Buckwalter-Default     buckwalter-default  42.0000 67.3000 1.6500
     Buckwalter-Norm-Dediac buckwalter-norm-dediac  43.0000 67.2000 1.6500
         Arabic-Nonorm-Diac     arabic-nonorm-diac  42.1000 66.6000 1.6700
             Arabic-Default         arabic-default  37.2000 66.1000 1.7600
Arabic-No-Unicode-Normalize   no-unicode-normalize  39.4000 65.4000 1.7300
         Arabic-Norm-Dediac     arabic-norm-dediac  38.2000 63.4000 1.8200

🥇 BEST PERFORMING MODEL
Model: Buckwalter-Nonorm-Diac
Processing: buckwalter-nonorm-diac
Accuracy: 42.8000
QWK: 68.4000
MAE: 1.6000


: 

## Export Results

Save results to files for further analysis:

In [8]:
import json
from datetime import datetime

# Create output directory
output_dir = "barec_evaluation_results"
os.makedirs(output_dir, exist_ok=True)

# Save detailed results as JSON
detailed_results = {
    'timestamp': datetime.now().isoformat(),
    'dataset_config': DATASET_CONFIG,
    'model_configs': [{
        'name': config.name,
        'model_path': config.model_path,
        'processing_config': config.processing_config,
        'description': config.description,
        'renderer_path': config.renderer_path
    } for config in MODEL_CONFIGS],
    'results': all_results
}

with open(f"{output_dir}/detailed_results.json", 'w', encoding='utf-8') as f:
    json.dump(detailed_results, f, ensure_ascii=False, indent=2, default=str)

# Save CSV files
validation_df.to_csv(f"{output_dir}/validation_results.csv", index=False)
test_df.to_csv(f"{output_dir}/test_results.csv", index=False)

if len(performance_df) > 0:
    performance_df.to_csv(f"{output_dir}/performance_ranking.csv", index=False)

print(f"💾 Results saved to '{output_dir}/' directory:")
print(f"   • detailed_results.json - Complete evaluation data")
print(f"   • validation_results.csv - Validation metrics table")
print(f"   • test_results.csv - Test predictions summary")
if len(performance_df) > 0:
    print(f"   • performance_ranking.csv - Model performance ranking")

💾 Results saved to 'barec_evaluation_results/' directory:
   • detailed_results.json - Complete evaluation data
   • validation_results.csv - Validation metrics table
   • test_results.csv - Test predictions summary
   • performance_ranking.csv - Model performance ranking


## Summary and Insights

Provide final summary and actionable insights:

In [9]:
print("📋 EVALUATION SUMMARY")
print("=" * 50)

# Count successful evaluations
successful_validations = len(validation_df[validation_df['Status'] == '✅ Success'])
successful_tests = len(test_df[test_df['Status'] == '✅ Success'])
total_models = len(MODEL_CONFIGS)

print(f"📊 Total Models Evaluated: {total_models}")
print(f"✅ Successful Validation Evaluations: {successful_validations}/{total_models}")
print(f"✅ Successful Test Evaluations: {successful_tests}/{total_models}")

if successful_validations > 0:
    # Performance insights
    val_success = validation_df[validation_df['Status'] == '✅ Success'].copy()
    
    # Convert to numeric for analysis
    for col in ['Accuracy', 'QWK', 'MAE']:
        val_success[col] = pd.to_numeric(val_success[col], errors='coerce')
    
    best_acc = val_success.loc[val_success['Accuracy'].idxmax()]
    best_qwk = val_success.loc[val_success['QWK'].idxmax()]
    best_mae = val_success.loc[val_success['MAE'].idxmin()]  # Lower is better
    
    print(f"\n🎯 PERFORMANCE HIGHLIGHTS")
    print(f"   Best Accuracy: {best_acc['Accuracy']:.4f} ({best_acc['Model']})")
    print(f"   Best QWK: {best_qwk['QWK']:.4f} ({best_qwk['Model']})")
    print(f"   Best MAE: {best_mae['MAE']:.4f} ({best_mae['Model']})")
    
    # Processing configuration insights
    config_performance = val_success.groupby('Processing Config')['Accuracy'].mean().sort_values(ascending=False)
    
    print(f"\n🔧 PROCESSING CONFIGURATION RANKING (by avg accuracy):")
    for i, (config, acc) in enumerate(config_performance.items(), 1):
        print(f"   {i}. {config}: {acc:.4f}")

print(f"\n💡 KEY INSIGHTS:")
if successful_validations > 0:
    print(f"   • Morphological processing configurations show varying performance")
    print(f"   • Consider the best performing configuration for production use")
    print(f"   • Tatweel replacement may improve visual consistency in rendering")
else:
    print(f"   • No successful evaluations - check model paths and configurations")
    print(f"   • Verify that models are accessible and compatible")

print(f"\n✨ Evaluation Complete! Check the output files for detailed results.")

📋 EVALUATION SUMMARY
📊 Total Models Evaluated: 10
✅ Successful Validation Evaluations: 10/10
✅ Successful Test Evaluations: 0/10

🎯 PERFORMANCE HIGHLIGHTS
   Best Accuracy: 0.4383 (Buckwalter-Default)
   Best QWK: 0.6801 (Buckwalter-Default)
   Best MAE: 1.6342 (Buckwalter-Default)

🔧 PROCESSING CONFIGURATION RANKING (by avg accuracy):
   1. buckwalter-default: 0.4383
   2. hsb-default: 0.4358
   3. buckwalter-norm-dediac: 0.4346
   4. buckwalter-nonorm-diac: 0.4320
   5. hsb-norm-dediac: 0.4111
   6. arabic-nonorm-diac: 0.4107
   7. hsb-nonorm-diac: 0.4063
   8. no-unicode-normalize: 0.3874
   9. arabic-norm-dediac: 0.3806
   10. arabic-default: 0.3565

💡 KEY INSIGHTS:
   • Morphological processing configurations show varying performance
   • Consider the best performing configuration for production use
   • Tatweel replacement may improve visual consistency in rendering

✨ Evaluation Complete! Check the output files for detailed results.
